In [2]:
!git config --global user.email "beglossy@yonsei.ac.kr"
!git config --global user.name "beglossy-cmyk"

from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/beglossy-cmyk/gpt2.0_by_InaYoon.git
%cd gpt2.0_by_InaYoon
print("완료!")

Cloning into 'gpt2.0_by_InaYoon'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 20 (delta 8), reused 11 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 13.70 KiB | 4.57 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/gpt2.0_by_InaYoon
완료!


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

if not Path("input.txt").exists():
    !wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

text = open("input.txt", "r", encoding="utf-8").read()
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

class NextTokenDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size
    def __len__(self):
        return len(self.data) - self.block_size
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

block_size = 32
dataset = NextTokenDataset(data, block_size)
loader = DataLoader(dataset, batch_size=64, shuffle=True)
xb, yb = next(iter(loader))

print(f"vocab_size: {vocab_size}")
print(f"xb.shape: {xb.shape}")

vocab_size: 65
xb.shape: torch.Size([64, 32])


In [4]:
class SingleHeadSelfAttention(nn.Module):
    def __init__(self, emb_dim, block_size):
        super().__init__()
        self.key   = nn.Linear(emb_dim, emb_dim, bias=False)
        self.query = nn.Linear(emb_dim, emb_dim, bias=False)
        self.value = nn.Linear(emb_dim, emb_dim, bias=False)
        # 미래를 보지 못하게 막는 삼각형 마스크
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, C)
        q = self.query(x)  # (B, T, C)
        v = self.value(x)  # (B, T, C)

        # Q × K → 점수 계산 (스케일링 포함)
        wei = q @ k.transpose(-2, -1) * (C ** -0.5)  # (B, T, T)
        # 미래 위치를 -inf로 마스킹
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        # softmax → 확률로 변환
        wei = F.softmax(wei, dim=-1)

        out = wei @ v  # (B, T, C)
        return out

# 테스트
attn = SingleHeadSelfAttention(emb_dim=64, block_size=32)
dummy = torch.randn(64, 32, 64)  # (B, T, C)
out = attn(dummy)
print(f"attention 출력 shape: {out.shape}")  # (64, 32, 64)

attention 출력 shape: torch.Size([64, 32, 64])


In [5]:
class TinyAttentionLM(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=64):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.attn = SingleHeadSelfAttention(emb_dim, block_size)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        tok = self.token_embedding(x)
        pos = self.position_embedding(torch.arange(T, device=x.device))
        h = tok + pos      # 글자 + 위치
        h = self.attn(h)   # Attention!
        logits = self.lm_head(h)
        return logits

model = TinyAttentionLM(vocab_size, block_size)
logits = model(xb)
print(f"logits.shape: {logits.shape}")  # (64, 32, 65)

logits.shape: torch.Size([64, 32, 65])


In [6]:
def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)

def train_one_epoch(model, loader, optimizer, device, max_steps=None):
    model.train()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"사용 device: {device}")

model = TinyAttentionLM(vocab_size, block_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

for epoch in range(100):
    train_loss = train_one_epoch(model, loader, optimizer, device, max_steps=300)
    if epoch % 10 == 0 or epoch == 99:
        print(f"epoch {epoch:2d} | train loss {train_loss:.4f}")

사용 device: cpu
epoch  0 | train loss 2.9131
epoch 10 | train loss 2.2965
epoch 20 | train loss 2.2740
epoch 30 | train loss 2.2610
epoch 40 | train loss 2.2594
epoch 50 | train loss 2.2532
epoch 60 | train loss 2.2511
epoch 70 | train loss 2.2491
epoch 80 | train loss 2.2442
epoch 90 | train loss 2.2454
epoch 99 | train loss 2.2449


In [7]:
@torch.no_grad()
def sample(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=300):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in start_text:
        if ch in stoi:
            ix = torch.tensor([[stoi[ch]]], device=device)
            context = torch.cat([context[:, 1:], ix], dim=1)
    out = list(start_text)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

print(sample(model, block_size, stoi, itos, device, start_text="ROMEO:", max_new_tokens=300))

ROMEO:
Sthis hant thou, bedeeveg asoss wike thucomee, tr!----ityo my enalpy
Is lod wimy the, thoith,
Yo abix sat yoth your afr the; bdons to yoff thy wert, Courd ise inse mofriath mat apeany, at ange wingin, to bronk he
The owlillas
u's ghhouses! ls savecoutarne hom
Whan atle msheangel se dy.

PERDI:e od 


In [ ]:
from google.colab import userdata, _message
import json

nb = _message.blocking_request('get_ipynb', request='', timeout_sec=120)
with open('/content/gpt2.0_by_InaYoon/notebook_05_attention.ipynb', 'w') as f:
    json.dump(nb['ipynb'], f)

token = userdata.get('GITHUB_TOKEN')
%cd /content/gpt2.0_by_InaYoon
!git remote set-url origin https://{token}@github.com/beglossy-cmyk/gpt2.0_by_InaYoon.git
!git add notebook_05_attention.ipynb
!git commit -m "Add notebook_05: Single-head Masked Self-Attention"
!git push origin main